# 필요 라이브러리 및 함수 정의

In [1]:
import time
import base64
import json
import csv
import random
import statistics

import numpy as np
import pandas as pd

In [2]:
def sliding_window(data, window_size, step):
    for start_row in range(0, len(data) - window_size + 1, step):
        yield data[start_row:start_row + window_size]        

# 데이터 전처리

In [3]:
def get_data(data, device_name):
    humid = []
    pm10 = []
    pm25 = []
    temp = []
    for i in range(len(data)):
        val = data.values[i][0]
        val = val.replace("'", "")
        val = val.replace("b","",1)
        json_val = json.loads(val)

        res_payload = json_val['Payload']
        dec_res = base64.b64decode(res_payload)
        dec_res = dec_res.decode("UTF-8")
        str_test = dec_res.replace("'","\"")
        json_data = json.loads(str_test)
        
        if json_data['event']['readings'][0]['deviceName'] == device_name:
            humid.append(json_data['event']['readings'][0]['objectValue']['humidity'])
            pm10.append(json_data['event']['readings'][0]['objectValue']['pm10'])
            pm25.append(json_data['event']['readings'][0]['objectValue']['pm25'])
            temp.append(json_data['event']['readings'][0]['objectValue']['temperature'])
    return humid, pm10, pm25, temp

# 변수별 차분 계산

In [4]:
def get_diff(humid, pm10, pm25, temp):
    #절대값 차분
    diff_humid = []
    diff_pm10 = []
    diff_pm25 = []
    diff_temp = []

    for i in range(len(humid)-1):
        diff_humid.append(abs(humid[i+1]-humid[i]))
        diff_pm10.append(abs(pm10[i+1]-pm10[i]))
        diff_pm25.append(abs(pm25[i+1]-pm25[i]))
        diff_temp.append(abs(temp[i+1]-temp[i]))
    return diff_humid, diff_pm10, diff_pm25, diff_temp

# 입력 데이터 정의

In [5]:
#file = input()

In [6]:
#test = pd.read_csv(file,header=None)

In [7]:
test = pd.read_csv("text_1.csv", header=None)
#test = pd.read_csv("text_2.csv", header=None)
#test = pd.read_csv("text_3.csv", header=None)
test = test.transpose()
#test

In [8]:
device_list = ['MD_01', 'MD_02', 'MD_03', 'MD_04', 'MD_05', 'MD_06', 'MD_07', 'MD_08', 'MD_09', 'MD_10']

# 감소율을 위한 최적값 계산

In [11]:
window_size = random.randint(9,19)
# step = random.randint(1,10)
weight = random.uniform(0.4,0.6)

total_count = 0
for i in range(10):
    device_humid, device_pm10, device_pm25, device_temp = get_data(test, device_list[i])
    device_humid_diff, device_pm10_diff, device_pm25_diff, device_temp_diff = get_diff(device_humid, device_pm10, device_pm25, device_temp)
    #print(device_humid_diff)
    
    humid_window = np.mean(device_humid_diff[0:window_size])
    pm10_window = np.mean(device_pm10_diff[0:window_size])
    pm25_window = np.mean(device_pm25_diff[0:window_size])
    temp_window = np.mean(device_temp_diff[0:window_size])
    
    device_humid_last = device_humid[window_size+1]
    device_pm10_last = device_pm10[window_size+1]
    device_pm25_last = device_pm25[window_size+1]
    device_temp_last = device_temp[window_size+1]
    
    cnt_val = []
    count = 0
    
    for index in range(window_size+1, 100):
        count = count + 1
        device_humid_input = device_humid[index]
        device_pm10_input = device_pm10[index]
        device_pm25_input = device_pm25[index]
        device_temp_input = device_temp[index]
        
        now_humid_diff = abs(device_humid_input - device_humid_last)
        now_pm10_diff = abs(device_pm10_input - device_pm10_last)
        now_pm25_diff = abs(device_pm25_input - device_pm25_last)
        now_temp_diff = abs(device_temp_input - device_temp_last)
        
        if now_humid_diff < humid_window * weight and now_pm10_diff < pm10_window * weight and now_pm25_diff < pm25_window * weight and now_temp_diff < temp_window * weight:
            cnt_val.append(index)
        else:
            device_humid_last = device_humid_input
            device_pm10_last = device_pm10_input
            device_pm25_last = device_pm25_input
            device_temp_last = device_temp_input
            
        if count > 0 and count % 10 == 0:
            loss_ratio = len(cnt_val) / count
#             print("입력 트래픽 수 :", count)
#             print("출력 트래픽 수 : ", count-len(cnt_val))
#             print("감소율 : ", round(loss_ratio * 100, 2), "%")
            if loss_ratio > 0.14:
                weight = weight - random.uniform(0.15,0.20)
            elif loss_ratio < 0.10:
                weight = weight + random.uniform(0.15,0.20)
#             print("변경된 가중치 : ", round(weight,2))
#             print('===========================')
                
    print(device_list[i], '===========================')
    loss_ratio = len(cnt_val) / count
    print("입력 트래픽 수 :", count)
    print("출력 트래픽 수 : ", count-len(cnt_val))
    print("감소율 : ", round(loss_ratio * 100, 2), "%")
    print(device_list[i], '===========================')
    
    total_count = total_count + len(cnt_val)

MD_01 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  80
감소율 :  5.88 %
MD_01 ===========================
MD_02 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  64
감소율 :  24.71 %
MD_02 ===========================
MD_03 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  76
감소율 :  10.59 %
MD_03 ===========================
MD_04 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  77
감소율 :  9.41 %
MD_04 ===========================
MD_05 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  74
감소율 :  12.94 %
MD_05 ===========================
MD_06 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  72
감소율 :  15.29 %
MD_06 ===========================
MD_07 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  75
감소율 :  11.76 %
MD_07 ===========================
MD_08 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  84
감소율 :  1.18 %
MD_08 ===========================
MD_09 ===========================
입력 트래픽 수 : 85
출력 트래픽 수 :  77
감소율 :  9.41 %
MD_09 ========================

In [12]:
loss_ratio = total_count / (1000 - window_size*10)
print("입력 트래픽 수 :", (1000 - window_size*10))
print("출력 트래픽 수 : ", (1000 - window_size*10)-total_count)
print("감소율 : ", round(loss_ratio * 100, 2), "%")

입력 트래픽 수 : 860
출력 트래픽 수 :  766
감소율 :  10.93 %
